In [4]:
from dotenv import load_dotenv
load_dotenv()
import os
from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI
import anthropic
import json

In [5]:
documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [12]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [13]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [14]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discover course can I join enrollment late registration FAQ"}
function_call: search {"query":"course discovered late can I join now FAQ registration"}
function_call: search {"query":"enroll join course after start FAQ"}


In [15]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still being accepted. Also, you can start learning and doing homework even without a confirmation email; registration is just for gauging interest.

If you want, I can also help with whether you can still get a certificate or how the course schedule works.


In [16]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [17]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local Ollama FAQ"}
function_call: search {"query":"run Ollama locally command line install model serve localhost"}
function_call: search {"query":"Ollama local setup FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**, do this:

1. **Install Ollama**
   - macOS: download from https://ollama.com/download and install the `.pkg`
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This downloads the model if needed and starts a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   If it’s running, you should get a JSON response.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "

'To run **Ollama locally**, do this:\n\n1. **Install Ollama**\n   - macOS: download from https://ollama.com/download and install the `.pkg`\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model if needed and starts a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s running, you should get a JSON response.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection error in the course notebook, you can restart the server with:\n\n```bash\nnohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you 

In [18]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I still join"}
function_call: search {"query":"course enrollment late join discovered course FAQ"}
function_call: search {"query":"can I still join the course after it started FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join.

If you want a certificate, make sure to submit your project while submissions are still being accepted. If you’re just looking to learn, you can start anytime.

If you’d like, I can also help with what you need to do next to catch up or how certificates work.


'Yes — you can still join.\n\nIf you want a certificate, make sure to submit your project while submissions are still being accepted. If you’re just looking to learn, you can start anytime.\n\nIf you’d like, I can also help with what you need to do next to catch up or how certificates work.'

In [19]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered late enrollment can I join course FAQ"}
iteration #2...
function_call: search {"query":"certificate join late submit project while accepting submissions peer review live cohort self-paced FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open, and the certificate is only available if you complete the course with the live cohort.

If you’d like, I can also help you figure out how the course works from here or what the next steps are. Any other areas you want to explore?


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open, and the certificate is only available if you complete the course with the live cohort.\n\nIf you’d like, I can also help you figure out how the course works from here or what the next steps are. Any other areas you want to explore?'

In [20]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess queen's gambit opening what is it"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening definition accepted declined"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the **c4 pawn** to try to control the center and get a strong position. It’s called a “gambit” because a pawn is offered early, though in many variations Black doesn’t actually keep the pawn for long.

Common responses from Black are:
- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`
- **Queen’s Gambit Declined**: Black does not take it and instead supports the center

If you want, I can also explain **why White plays c4** or show a few **basic Queen’s Gambit lines**.


'The **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c4 pawn** to try to control the center and get a strong position. It’s called a “gambit” because a pawn is offered early, though in many variations Black doesn’t actually keep the pawn for long.\n\nCommon responses from Black are:\n- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`\n- **Queen’s Gambit Declined**: Black does not take it and instead supports the center\n\nIf you want, I can also explain **why White plays c4** or show a few **basic Queen’s Gambit lines**.'

In [21]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit queen's gambit course FAQ"}
iteration #2...
function_call: search {"query":"queen gambit gambit queen course FAQ chess"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen gambit,” so it looks like this isn’t a course-related question.

If you meant something else from the course or logistics, feel free to rephrase it. Are there other areas you want to explore?


'I couldn’t find any course FAQ entry about “queen gambit,” so it looks like this isn’t a course-related question.\n\nIf you meant something else from the course or logistics, feel free to rephrase it. Are there other areas you want to explore?'